# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder

## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [2]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)

14 specific + 10 comparative = 24 labeled questions
reranker:       cohere
chat model:     gemini-3.5-flash-lite
decision model: clef-flash


## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

scoper = config.chat().with_structured_output(Scope)
# your code here

In [4]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

decision  20/24 right, 0.99s per call
chat      18/24 right, 0.88s per call


NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [5]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)

  agent         accuracy  chat/q  decision/q    s/q
  --------------------------------------------------


c:\Users\reyaa\agentic-ai-course\DADA2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ladder            0.86     3.9         0.0   12.6
  diagnosis         0.86     2.9         2.4   11.7


In [ ]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

In [7]:
for name, fn in [("ladder", ladder.run), ("diagnosis", agent.run)]:
    print(f"=== {name} ===")
    _, notes = evaluate.score_agent(fn, comparative[:6])
    for note in notes:
        mark = "ok   " if note["correct"] else "WRONG"
        print(f"[{mark}] {note['question'][:60]}  ({note['chat_calls']} chat)")
    print()

=== ladder ===
[ok   ] How has NVIDIA's exposure to export controls changed over th  (3 chat)
[ok   ] Which of these companies treat dependence on a small number   (9 chat)
[ok   ] What do they say about relying on third-party foundries to m  (3 chat)
[ok   ] How do AMD and Micron each describe competition in their mos  (9 chat)
[ok   ] Is anyone worried about China?  (3 chat)
[ok   ] What does Micron say about memory pricing and how it moves?  (3 chat)

=== diagnosis ===
[ok   ] How has NVIDIA's exposure to export controls changed over th  (2 chat)
[ok   ] Which of these companies treat dependence on a small number   (2 chat)
[ok   ] What do they say about relying on third-party foundries to m  (3 chat)
[ok   ] How do AMD and Micron each describe competition in their mos  (5 chat)
[ok   ] Is anyone worried about China?  (2 chat)
[ok   ] What does Micron say about memory pricing and how it moves?  (2 chat)



In [6]:
rows

[{'agent': 'ladder',
  'accuracy': 0.8571428571428571,
  'chat_calls': 3.857142857142857,
  'decision_calls': 0.0,
  'seconds': 12.563477907861982,
  'n': 14},
 {'agent': 'diagnosis',
  'accuracy': 0.8571428571428571,
  'chat_calls': 2.857142857142857,
  'decision_calls': 2.357142857142857,
  'seconds': 11.7101491689682,
  'n': 14}]

If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


---

### Your write-up

*(replace this cell)*

**Conditions.**

**The routers.**

**The agents.**

**What did not help.**

**What I would ship.**

**Copy your write-up to README.md.**

Conditions. The corpus is 27 section files (Business, Risk Factors and MD&A) from nine 10-K filings, three years each for NVIDIA, AMD and Micron, chunked at 1,200 characters with 150 characters of overlap into 2,124 chunks tagged with source, ticker, fiscal year and section. The chunks are embedded with qwen3-embedding-0.6b on Cloudflare Workers AI (1,024 dimensions) and stored in the Pinecone index dada2-1024. The chat model is gemini-3.5-flash-lite, the decision model is clef-flash on Cloudflare and the reranker is Cohere rerank-v3.5; each search returns k = 5 passages, and the reranking strategy reorders a pool of 10, with an RRF constant of 60. The routers were tested on all 24 labeled questions (14 specific, 10 comparative) and the agents on 14 (8 specific, 6 comparative), with the chat model judging each answer correct or not.

The routers. The decision model was more accurate, routing 20 of 24 questions correctly (83%) against Gemini’s 18 (75%), while Gemini was faster by about a tenth of a second per call (0.88 s against 0.99 s), which is negligible. That two-question gap may not hold, though: the notebook’s reference run had Gemini slightly ahead, and with 24 questions and a non-deterministic chat model the order can flip between runs. What the decision model adds is a probability for each option, which lets route fall back to DEFAULT below SURE = 0.6, plus calls that don’t count against Gemini’s 500-a-day limit. Micron’s R&D question shows what the probabilities add: it came back 0.63 for figure and 0.36 for specific, just over the threshold, so it still routed to figure, but it was a close call that a single label from Gemini wouldn’t show.

The agents. Both agents scored 0.86, but the diagnosis agent made 2.9 chat calls per question (plus 2.4 decision-model calls, which don’t count against the Gemini limit) to the ladder’s 3.9 and took 11.7 s per question to its 12.6 s: the same accuracy for about a quarter fewer Gemini calls, so this is a cost improvement, not an accuracy one. The question where they differed most came from a second run on the six comparative questions: “Which of these companies treat dependence on a small number of customers as a risk?” took the ladder 9 chat calls and diagnosis 2. Because the question names no company, 9 calls is consistent with the ladder’s yes/no grader judging each rung insufficient, so it kept climbing, and each climb cost a grader call plus the search’s own calls. Diagnosis routed it as comparative to hybrid+rrf in one call, the decision model judged those passages sufficient without a Gemini call, and it answered in one more. Across the six comparative questions the totals were 30 chat calls to 16, close to half, so the savings come from comparative questions; specific ones are cheap for both agents.

What did not help. The metadata filter did not help on comparative questions: on a 5-question comparative sample it cut recall@10 from 0.83 without the filter to 0.63 with it, even though the same filter had raised recall@5 on the 6-question specific sample from 0.67 to 1.00. The mechanism is in where_of, which restricts the search to the one company or year the extractor picks and so throws away the other companies’ filings that a comparative question needs. The design routes around it: the router sends comparative questions to hybrid+rrf, and the planner splits them into one filtered search per company.

What I would ship. I would ship the diagnosis agent, because it matched the ladder’s accuracy with fewer Gemini calls. Before shipping I’d want to know the mix of questions users ask: if most are comparative the savings are large, and if most are specific the two agents cost about the same. I’d also want to know how often they ask number questions or about things the filings don’t cover, since the second is where the ladder can waste the most climbs while diagnosis stops after one search (the dress-code question).